# Double Overtime?

## I. Summary
Here, we search for officers who engage in two overtime shifts simultaneously;  
We find that there were likely no such instances.

## II. Importing libraries and datasets

In [54]:
import pandas as pd
from datetime import datetime

In [55]:
df_court = pd.read_csv("../cleaned_data/Court_merged_cleaned.csv")
df_detail = pd.read_csv("../cleaned_data/Detail_merged_cleaned.csv")
df_events = pd.read_csv("../cleaned_data/Events_merged_cleaned.csv")
df_records = pd.read_csv("../cleaned_data/Records_merged_cleaned.csv")

## III. Formatting datasets

In [56]:
df_court.columns, df_detail.columns, df_events.columns, df_records.columns

(Index(['IDNO6', 'NAME', 'RANK', 'ASSIGNED_DESC', 'CHARGED_DESC', 'OTDATE',
        'STARTTIME', 'ENDTIME', 'WRKDHRS'],
       dtype='object'),
 Index(['RowId', 'Tracking_No', 'Emp_No', 'Emp_Name', 'Emp_Rank',
        'Emp_Org_Code', 'District_Worked', 'Customer_No', 'Customer_Seq',
        'Customer_No_And_Seq', 'BPD_Customer_no', 'Street', 'Location',
        'Start_date', 'Start_Time', 'End_Time', 'Hours_Worked', 'Detail_Type',
        'Detail_Rank', 'Admin_Fee_Flag', 'Pay_Hours', 'Pay_Amount',
        'Pay_TRC_Code', 'Pay_Rate', 'FBK_Pay_Date'],
       dtype='object'),
 Index(['ID', 'NAME', 'RANK', 'ASSIGNED_DESC', 'CHARGED_DESC', 'OTDATE',
        'DESCRIPTION', 'STARTTIME', 'ENDTIME', 'OTHOURS'],
       dtype='object'),
 Index(['IDNO6', 'NAME', 'RANK', 'ASSIGNED_DESC', 'CHARGED_DESC', 'OTDATE',
        'DESCRIPTION', 'STARTTIME', 'ENDTIME', 'WRKDHRS', 'OTHOURS'],
       dtype='object'))

In [57]:
df_detail.head()

,RowId,Tracking_No,Emp_No,Emp_Name,Emp_Rank,Emp_Org_Code,District_Worked,Customer_No,Customer_Seq,Customer_No_And_Seq,...,End_Time,Hours_Worked,Detail_Type,Detail_Rank,Admin_Fee_Flag,Pay_Hours,Pay_Amount,Pay_TRC_Code,Pay_Rate,FBK_Pay_Date
0,730807,2206472,11173,"HARDY,JOHN J",5,41070.0,7,99163,1,00099163-1,...,1600,830,C,5,N,900,495,P05DC,55,1/10/2020
1,730828,2206484,9755,"HORTON,JOSEPH R",5,45130.0,13,99163,1,00099163-1,...,1600,830,C,5,N,900,495,P05DC,55,1/10/2020
2,730831,2206473,11612,"HYNES,JOHN F",7,43060.0,6,99163,1,00099163-1,...,1600,830,C,7,N,900,477,P07DC,53,1/10/2020
3,730970,2207457,10745,"WILSON,MICHAEL C",9,11060.0,1,95347,1,095347-1,...,500,515,C,9,Y,800,368,P09DC,46,1/17/2020
4,731041,2210719,12130,"MARANDO,SHAWN P",9,41070.0,7,97297,1,097297-1,...,2100,500,C,9,Y,800,368,P09DC,46,1/17/2020


In [58]:
df_detail1 = df_detail[['Emp_No', 'Emp_Name', 'Emp_Rank', 'Start_date', 'Start_Time', 'End_Time']]
df_detail1 = df_detail1.rename(columns = {'Emp_No': 'IDNO6', 'Start_date': 'OTDATE', 'Start_Time': 'STARTTIME', 'End_Time': 'ENDTIME'})

In [59]:
df_court['OTDATE'] = pd.to_datetime(df_court['OTDATE']).dt.strftime('%Y-%m-%d')
df_events['OTDATE'] = pd.to_datetime(df_events['OTDATE']).dt.strftime('%Y-%m-%d')

C:\Users\taiyo\AppData\Local\Temp\ipykernel_9312\2716188621.py:2: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_events['OTDATE'] = pd.to_datetime(df_events['OTDATE']).dt.strftime('%Y-%m-%d')


In [60]:
df_court1 = df_court.rename(columns={'WRKDHRS': 'OTHOURS'})
df_events1 = df_events.drop(['DESCRIPTION'], axis=1).rename(columns={'ID': 'IDNO6'})
df_records1 = df_records.drop(['DESCRIPTION', 'WRKDHRS'], axis=1)
#df_combined = pd.concat([df_court1, df_events1, df_records1])

In [61]:
df_court1.shape, df_events1.shape, df_records1.shape

((39750, 9), (42196, 9), (406022, 9))

In [62]:
df_court1.loc[df_court1["OTHOURS"] < 4.0, "ENDTIME"] = df_court1["STARTTIME"] + 400

In [63]:
df_court1.head()

,IDNO6,NAME,RANK,ASSIGNED_DESC,CHARGED_DESC,OTDATE,STARTTIME,ENDTIME,OTHOURS
0,93855.0,"Rodrigues,Joao C",Ptl,YOUTH VIOLENCE STRIKE FORCE,YOUTH VIOLENCE STRIKE FORCE,2020-01-03,830,1230,1.0
1,11894.0,"Cutone,Anthony",Det,D-14 DETECTIVES,D-14 DETECTIVES,2020-01-02,1000,1400,0.5
2,131212.0,"O'Loughlin,James M",Ptl,DISTRICT 02,DISTRICT 02,2020-01-03,900,1300,1.5
3,131218.0,"Smerz,Christopher",Ptl,DISTRICT 02,DISTRICT 02,2020-01-03,830,1230,0.5
4,103632.0,"Finn,Kevin P",Det,C-11 DETECTIVES,C-11 DETECTIVES,2020-01-03,1530,1930,0.5


In [64]:
#df_sameday = df_combined[df_combined.duplicated(subset=['IDNO6', 'OTDATE'], keep=False)].sort_values(['IDNO6', 'OTDATE', 'STARTTIME'])

## IV. Searching for double dipping

In [65]:
def check_overlap(start1, start2, end1, end2):
    return (start1 <= end2 and end1 >= start2) #or (start1 <= end2 and end1 > start2)

In [66]:
df_court1.dtypes, df_events1.dtypes, df_records1.dtypes

(IDNO6            float64
 NAME              object
 RANK              object
 ASSIGNED_DESC     object
 CHARGED_DESC      object
 OTDATE            object
 STARTTIME          int64
 ENDTIME            int64
 OTHOURS          float64
 dtype: object,
 IDNO6            float64
 NAME              object
 RANK              object
 ASSIGNED_DESC     object
 CHARGED_DESC      object
 OTDATE            object
 STARTTIME        float64
 ENDTIME          float64
 OTHOURS          float64
 dtype: object,
 IDNO6            float64
 NAME              object
 RANK              object
 ASSIGNED_DESC     object
 CHARGED_DESC      object
 OTDATE            object
 STARTTIME          int64
 ENDTIME            int64
 OTHOURS          float64
 dtype: object)

In [67]:
df_court1['STARTTIME'] = df_court1['STARTTIME'].astype(float)
df_court1['ENDTIME'] = df_court1['ENDTIME'].astype(float)

df_records1['STARTTIME'] = df_records1['STARTTIME'].astype(float)
df_records1['ENDTIME'] = df_records1['ENDTIME'].astype(float)

### Court and Special Events
*Note: The code below is partially from generative AI*

In [68]:
# Define a function to find overlaps within each group for df_court1 and df_events1
def find_overlaps(group):
    results = []
    # Compare each row in df_court1 with rows in df_events1
    for _, row_court in group['df_court1'].iterrows():
        for _, row_event in group['df_events1'].iterrows():
            # Check for overlap between df_court1 and df_events1
            if check_overlap(row_court["STARTTIME"], row_event["STARTTIME"], row_court["ENDTIME"], row_event["ENDTIME"]):
                results.append({
                    "IDNO6": row_court["IDNO6"],
                    "OTDATE": row_court["OTDATE"],
                    "court Start": row_court["STARTTIME"],
                    "court End": row_court["ENDTIME"],
                    "event Start": row_event["STARTTIME"],
                    "event End": row_event["ENDTIME"]
                })
    return pd.DataFrame(results)

# Group each dataframe by "IDNO6" and "OTDATE" and convert to dictionary of DataFrames
court_groups = {name: group for name, group in df_court1.groupby(["IDNO6", "OTDATE"])}
event_groups = {name: group for name, group in df_events1.groupby(["IDNO6", "OTDATE"])}

# Iterate over each group key (IDNO6, OTDATE) to find overlaps
overlapping_results = []
for key in court_groups.keys():
    if key in event_groups:
        group = {
            'df_court1': court_groups[key],
            'df_events1': event_groups[key]
        }
        overlapping_results.append(find_overlaps(group))

# Combine all overlapping results into a single DataFrame
overlapping_timespans = pd.concat(overlapping_results, ignore_index=True)

In [69]:
overlapping_timespans

,IDNO6,OTDATE,court Start,court End,event Start,event End
0,11715.0,2022-06-24,1000.0,1400.0,1400.0,1600.0
1,11715.0,2022-06-30,1000.0,1400.0,1400.0,1600.0
2,11982.0,2022-07-13,830.0,1230.0,1230.0,1600.0
3,12028.0,2022-06-13,830.0,1500.0,1500.0,1600.0
4,12125.0,2021-09-24,900.0,1300.0,1300.0,2345.0
5,12232.0,2022-06-16,1400.0,1800.0,1000.0,1400.0
6,53603.0,2022-05-09,1200.0,1600.0,1600.0,2345.0
7,56151.0,2022-05-27,830.0,1230.0,1230.0,1500.0
8,56151.0,2022-06-08,900.0,1300.0,1300.0,1600.0
9,56151.0,2022-06-24,1000.0,1400.0,1400.0,1600.0


### Court, Special Events, and Records Request
Records Request contains some rows of `df_court1` and `df_events1`, so the code below may flag these as overlapping.

In [71]:
# Define a function to find overlaps within each group for at least two tables
def find_overlaps(group):
    results = []
    # Compare each row in df_court1 with rows in df_events1 and df_records1
    for _, row_court in group['df_court1'].iterrows():
        for _, row_event in group['df_events1'].iterrows():
            # Check for overlap between df_court1 and df_events1
            if check_overlap(row_court["STARTTIME"], row_event["STARTTIME"], row_court["ENDTIME"], row_event["ENDTIME"]):
                results.append({
                    "IDNO6": row_court["IDNO6"],
                    "OTDATE": row_court["OTDATE"],
                    "source": "court & events",
                    "court Start": row_court["STARTTIME"],
                    "court End": row_court["ENDTIME"],
                    "event Start": row_event["STARTTIME"],
                    "event End": row_event["ENDTIME"]
                })

        for _, row_record in group['df_records1'].iterrows():
            # Check for overlap between df_court1 and df_records1
            if check_overlap(row_court["STARTTIME"], row_record["STARTTIME"], row_court["ENDTIME"], row_record["ENDTIME"]):
                results.append({
                    "IDNO6": row_court["IDNO6"],
                    "OTDATE": row_court["OTDATE"],
                    "source": "court & records",
                    "court Start": row_court["STARTTIME"],
                    "court End": row_court["ENDTIME"],
                    "record Start": row_record["STARTTIME"],
                    "record End": row_record["ENDTIME"]
                })

        for _, row_event in group['df_events1'].iterrows():
            for _, row_record in group['df_records1'].iterrows():
                # Check for overlap between df_events1 and df_records1
                if check_overlap(row_event["STARTTIME"], row_record["STARTTIME"], row_event["ENDTIME"], row_record["ENDTIME"]):
                    results.append({
                        "IDNO6": row_event["IDNO6"],
                        "OTDATE": row_event["OTDATE"],
                        "source": "events & records",
                        "event Start": row_event["STARTTIME"],
                        "event End": row_event["ENDTIME"],
                        "record Start": row_record["STARTTIME"],
                        "record End": row_record["ENDTIME"]
                    })
    return pd.DataFrame(results)

# Group each dataframe by "IDNO6" and "OTDATE" and convert to dictionary of DataFrames
court_groups = {name: group for name, group in df_court1.groupby(["IDNO6", "OTDATE"])}
event_groups = {name: group for name, group in df_events1.groupby(["IDNO6", "OTDATE"])}
record_groups = {name: group for name, group in df_records1.groupby(["IDNO6", "OTDATE"])}

# Iterate over each group key (IDNO6, OTDATE) to find overlaps
overlapping_results = []
for key in court_groups.keys():
    if key in event_groups and key in record_groups:
        group = {
            'df_court1': court_groups[key],
            'df_events1': event_groups[key],
            'df_records1': record_groups[key]
        }
        overlapping_results.append(find_overlaps(group))

# Combine all overlapping results into a single DataFrame
overlapping_timespans = pd.concat(overlapping_results, ignore_index=True)


In [72]:
overlapping_timespans

,IDNO6,OTDATE,source,court Start,court End,record Start,record End,event Start,event End
0,9014.0,2020-02-10,court & records,830.0,1230.0,830.0,1000.0,NaN,NaN
1,10090.0,2020-01-24,court & records,830.0,1230.0,830.0,900.0,NaN,NaN
2,10090.0,2020-01-24,events & records,NaN,NaN,1600.0,2345.0,1600.0,2345.0
3,10090.0,2020-03-13,court & records,1200.0,1600.0,1200.0,1300.0,NaN,NaN
4,10396.0,2020-02-07,court & records,930.0,1330.0,930.0,1030.0,NaN,NaN
...,...,...,...,...,...,...,...,...,...
180,157695.0,2021-08-06,events & records,NaN,NaN,1600.0,2345.0,1600.0,2345.0
181,162930.0,2021-12-29,court & events,900.0,1300.0,NaN,NaN,1300.0,1600.0
182,162930.0,2021-12-29,court & records,900.0,1300.0,900.0,930.0,NaN,NaN
183,162930.0,2021-12-29,court & records,900.0,1300.0,1300.0,1600.0,NaN,NaN


### Records Request

In [73]:
def check_overlap(start1, start2, end1, end2):
    return (start1 < end2 and end1 > start2) #or (start1 <= end2 and end1 > start2)

# Define a function to find overlaps within each group in df_records1
def find_internal_overlaps(group):
    results = []
    # Compare each row with every other row in the same group
    for i, row1 in group.iterrows():
        for j, row2 in group.iterrows():
            if i >= j:  # Avoid double-counting and self-comparison
                continue
            # Check for overlap within df_records1
            if check_overlap(row1["STARTTIME"], row2["STARTTIME"], row1["ENDTIME"], row2["ENDTIME"]):
                results.append({
                    "IDNO6": row1["IDNO6"],
                    "OTDATE": row1["OTDATE"],
                    "Start1": row1["STARTTIME"],
                    "End1": row1["ENDTIME"],
                    "Start2": row2["STARTTIME"],
                    "End2": row2["ENDTIME"]
                })
    return pd.DataFrame(results)

# Group df_records1 by "IDNO6" and "OTDATE" and check for internal overlaps in each group
overlapping_results = []
for name, group in df_records1.groupby(["IDNO6", "OTDATE"]):
    overlapping_results.append(find_internal_overlaps(group))

# Combine all overlapping results into a single DataFrame
overlapping_timespans = pd.concat(overlapping_results, ignore_index=True)


In [74]:
overlapping_timespans

""


We can infer from the above `overlapping_timespans` being empty that there were no instances of double-dipping across overtime shifts.

### Detail

In [76]:
# Define a function to find overlaps within each group for df_court1 and df_events1
def find_overlaps(group):
    results = []
    # Compare each row in df_court1 with rows in df_events1
    for _, row_records in group['df_records1'].iterrows():
        for _, row_detail in group['df_detail1'].iterrows():
            # Check for overlap between df_court1 and df_events1
            if check_overlap(row_records["STARTTIME"], row_detail["STARTTIME"], row_records["ENDTIME"], row_detail["ENDTIME"]):
                results.append({
                    "IDNO6": row_records["IDNO6"],
                    "OTDATE": row_records["OTDATE"],
                    "records Start": row_records["STARTTIME"],
                    "records End": row_records["ENDTIME"],
                    "detail Start": row_detail["STARTTIME"],
                    "detail End": row_detail["ENDTIME"]
                })
    return pd.DataFrame(results)

# Group each dataframe by "IDNO6" and "OTDATE" and convert to dictionary of DataFrames
record_groups = {name: group for name, group in df_court1.groupby(["IDNO6", "OTDATE"])}
detail_groups = {name: group for name, group in df_events1.groupby(["IDNO6", "OTDATE"])}

# Iterate over each group key (IDNO6, OTDATE) to find overlaps
overlapping_results = []
for key in record_groups.keys():
    if key in detail_groups:
        group = {
            'df_records1': record_groups[key],
            'df_detail1': detail_groups[key]
        }
        overlapping_results.append(find_overlaps(group))

# Combine all overlapping results into a single DataFrame
overlapping_timespans = pd.concat(overlapping_results, ignore_index=True)

In [77]:
overlapping_timespans

""
